# Oracles and Smart Contracts — The Notary Learns To Fill In A Form

Up to now the chain has been a notary. You hand it something, it stamps it, it files it, it never reads it. The `data` field held a string. The chain had no opinion about what that string meant.

A smart contract is what happens when the notary agrees to fill in a form rather than just stamp one.

Notebook 2 gave us validators. Notebook 5 gave us a waiting room that is not one global queue. We **import** those classes. We do not reinvent them. We give them something to run.

**Question:** if a contract is a brain in a jar — perfect memory of its own state, no eyes, no windows — how does a lending protocol know what collateral is worth?

**Scope:** a deterministic teaching model, not a recipe for shipping or attacking a real protocol. No bytecode, no gas schedule, no lawyers. Same public PoS chain as notebooks 2 and 5, now filling in forms.

The toy model is ours. The mechanisms are not — see **Sources** at the end.


## Recap

Book One (notebooks 1–3) was a driving lesson in an empty car park: one laptop, one chain object, one truth, no friends. Notebook 5 admitted the internet exists.

| Notebook | What it established |
| --- | --- |
| 2 | Validators, a stake-weighted proposer, slashing as hostage. |
| 5 | Each node has its own mempool. `broadcast`, then `include`. The chain is a notary, not a psychic. |

**KEY INSIGHT.** Agreement does not create information. A blockchain can make a shared history hard to rewrite. It cannot make four honest laptops receive the same gossip at the same millisecond — and it cannot make a contract see Coinbase.

The new classes below — `SmartContract`, `AMMPool`, `LendingProtocol`, `PriceFeed`, `MedianOracle` — are defined inline so the lesson stays in one place. Notebook 7 imports the shared copy from [`blockchain_lib/contracts.py`](../blockchain_lib/contracts.py). Same handoff as mempool after notebook 5.


## 1. What a smart contract actually is

Because this is one of those concepts where one definition never sticks, keep whichever version survives:

- **The vending machine.** Put in $2, press B4, get a Snickers. No manager to escalate to. Deployed at a public address instead of a shopping mall.
- **The technical one.** Code at a permanent address, plus storage. Anyone can call a method. Every full node re-executes and must get the same answer.
- **The bureaucracy.** A form so rigid that ten thousand clerks in different countries produce identical results without consulting each other. Which, said that way, is the most terrifying office in the world.

On Ethereum the rules would be bytecode. Here they are Python, because we are trying to remember an idea, not compile Solidity in a cafe.

`SmartContract` is deliberately thin: an `address`, and `call(method, **kwargs)`. Private names (`_...`) are not part of the surface. Three contracts will inherit it — a swap pool, a lending protocol, and a price feed — because those are the forms that will live at addresses on the chain we already built.

**KEY INSIGHT.** "Smart contracts are immutable" is true of the **code** and false of the **state**. The rules cannot be edited. The numbers those rules operate on move constantly. If you can move the numbers, you can steer a perfectly honest, unhackable, unmodifiable contract straight off a cliff without breaking a single rule.

> Pause and predict: if the notary fills in the form correctly, does that mean the form was a good idea?


In [ ]:
import random
import statistics
from dataclasses import dataclass
from typing import Any

from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Block, Blockchain, Validator


class SmartContract:
    """Code that lives at an address and runs when something calls it.

    The chain does not understand loans or swaps. It understands: this
    address, this method, these arguments. The methods below are the rules.
    """

    def __init__(self, address: str) -> None:
        """Store the address this contract will be called at.

        Args:
            address: Non-empty on-chain name, e.g. ``amm.eth``.

        Raises:
            ValueError: If ``address`` is empty.
        """
        if not address:
            raise ValueError("Contract address must be non-empty.")
        self.address = address

    def call(self, method: str, **kwargs: Any) -> Any:
        """Invoke a public method by name.

        Args:
            method: Public method name. Names starting with ``_`` are
                not part of the callable surface.
            **kwargs: Arguments forwarded to that method.

        Returns:
            Whatever the underlying method returns.

        Raises:
            AttributeError: If ``method`` is private or missing.
        """
        if method.startswith("_") or method == "call":
            raise AttributeError(
                f"{self.address} has no public method {method!r}."
            )
        func = getattr(self, method, None)
        if not callable(func):
            raise AttributeError(  # noqa: TRY004
                f"{self.address} has no public method {method!r}."
            )
        return func(**kwargs)


def submit_call(
    network: Network,
    chain: Blockchain,
    origin: str,
    tx_id: str,
    contract: SmartContract,
    method: str,
    **kwargs: Any,
) -> tuple[Any, Transaction, Block]:
    """"Submit a contract call as a transaction.

    The toy network gossips the transaction, includes it in a block,
    and then executes the contract call. Real blockchain execution
    semantics are more nuanced; this simplified ordering is used to
    make the flow easy to follow.

    Args:
        network: Gossip network whose local mempools will see the call.
        chain: Chain that will record the inclusion.
        origin: Node that first broadcasts the transaction.
        tx_id: Unique identifier for this call.
        contract: Contract to invoke after inclusion.
        method: Public method name on ``contract``.
        **kwargs: Arguments forwarded to that method.

    Returns:
        ``(result, transaction, block)`` — whatever the method returned,
        the gossiped transaction, and the block that included it.
    """
    arg_preview = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
    description = f"{contract.address}.{method}({arg_preview})"
    network.broadcast(Transaction(tx_id, description), origin=origin)
    tx, block = network.include(origin, tx_id, chain)
    result = contract.call(method, **kwargs)
    return result, tx, block


## 2. The AMM, or a puddle with opinions about the ocean

Normal markets match buyers to sellers via an order book. That needs a server, a company, and someone to run it. Everything we are trying to avoid.

So: a contract holding two pots of tokens that quotes prices from nothing but the ratio of those two pots. The rule is one line. `x * y = k`. Spot price is `USD reserve / ETH reserve`. That number is not a journalist. It is the current ratio of two piles of tokens.

The next cell defines `AMMPool` — the same class notebook 7 will import from [`blockchain_lib/contracts.py`](../blockchain_lib/contracts.py). After that, **FROM THE LAB:** three identical puddles, 50 ETH and $100,000 each ($2,000/ETH). Drop a pebble in one. Drop a boulder in another.

**ANALOGY.** A pool is a puddle. Drop a pebble and you get a ripple. Drop a boulder and you displace half the water and soak everyone standing nearby. Neither event tells you anything whatsoever about the ocean. The puddle is reporting honestly about the puddle.

> Pause and predict: which sale moves the displayed price more, and does `x * y` remain essentially unchanged?


In [5]:
class AMMPool(SmartContract):
    """A constant-product (x*y=k) two-asset market maker.

    This is a smart contract: it has an address, and swaps are method
    calls. The displayed spot price is ``USD reserve / ETH reserve``.
    That number is not a journalist. It is the current ratio of two
    piles of tokens.
    """

    def __init__(
        self, address: str, eth_reserve: float, usd_reserve: float
    ) -> None:
        """Create a pool at ``address`` with the given reserves.

        Args:
            address: On-chain address of this pool.
            eth_reserve: ETH held by the pool. Must be positive.
            usd_reserve: USD held by the pool. Must be positive.

        Raises:
            ValueError: If either reserve is not positive.
        """
        super().__init__(address)
        if eth_reserve <= 0 or usd_reserve <= 0:
            raise ValueError("AMM reserves must be positive.")
        self.eth_reserve = eth_reserve
        self.usd_reserve = usd_reserve

    @property
    def spot_price(self) -> float:
        """Current displayed price: USD reserve per unit of ETH reserve."""
        return self.usd_reserve / self.eth_reserve

    @property
    def constant_product(self) -> float:
        """The invariant ``x * y`` a swap should preserve (no fees here)."""
        return self.eth_reserve * self.usd_reserve

    def swap_eth_for_usd(self, eth_in: float) -> float:
        """Sell ETH into the pool, moving both reserves and the spot price.

        Args:
            eth_in: ETH sold into the pool. Must be positive.

        Returns:
            USD received in exchange.

        Raises:
            ValueError: If ``eth_in`` is not positive.
        """
        if eth_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_eth_reserve = self.eth_reserve + eth_in
        new_usd_reserve = k / new_eth_reserve
        usd_out = self.usd_reserve - new_usd_reserve
        self.eth_reserve, self.usd_reserve = new_eth_reserve, new_usd_reserve
        return usd_out

    def swap_usd_for_eth(self, usd_in: float) -> float:
        """Sell USD into the pool, moving both reserves and the spot price.

        Args:
            usd_in: USD sold into the pool. Must be positive.

        Returns:
            ETH received in exchange.

        Raises:
            ValueError: If ``usd_in`` is not positive.
        """
        if usd_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_usd_reserve = self.usd_reserve + usd_in
        new_eth_reserve = k / new_usd_reserve
        eth_out = self.eth_reserve - new_eth_reserve
        self.usd_reserve, self.eth_reserve = new_usd_reserve, new_eth_reserve
        return eth_out


In [6]:
# Three identical puddles. Share one and the boulder contaminates the pebble's crime scene.
initial_pool = AMMPool("amm-initial", 50.0, 100_000.0)
small_trade_pool = AMMPool("amm-pebble", 50.0, 100_000.0)
large_trade_pool = AMMPool("amm-boulder", 50.0, 100_000.0)

# call() is the vending-machine button. Later a Block will press it; here we press it ourselves.
small_usd_out = small_trade_pool.call("swap_eth_for_usd", eth_in=1.0)
large_usd_out = large_trade_pool.call("swap_eth_for_usd", eth_in=40.0)
small_impact = (small_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
large_impact = (large_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
constant_product_preserved = abs(
    large_trade_pool.constant_product - initial_pool.constant_product
) < 1e-6

print(
    f"Initial pool: {initial_pool.eth_reserve:.2f} ETH and "
    f"${initial_pool.usd_reserve:,.2f}; spot price ${initial_pool.spot_price:,.2f}/ETH"
)
print(
    f"Small 1 ETH sale: receives ${small_usd_out:,.2f}; "
    f"price ${small_trade_pool.spot_price:,.2f}/ETH ({small_impact:.2f}%)"
)
print(
    f"Large 40 ETH sale: receives ${large_usd_out:,.2f}; "
    f"price ${large_trade_pool.spot_price:,.2f}/ETH ({large_impact:.2f}%)"
)
print(f"x * y preserved within floating-point tolerance: {constant_product_preserved}")


Initial pool: 50.00 ETH and $100,000.00; spot price $2,000.00/ETH
Small 1 ETH sale: receives $1,960.78; price $1,922.34/ETH (-3.88%)
Large 40 ETH sale: receives $44,444.44; price $617.28/ETH (-69.14%)
x * y preserved within floating-point tolerance: True


**Read the result.** Selling 1 ETH got you a ripple. Selling 40 ETH got you $44,444.44 and a displayed price of about **$617/ETH** — a 69% crash of this one puddle. `k` did not budge. There is no code path that could make it budge. The invariant is not maintained by anyone; it is enforced by construction.

**KEY INSIGHT.** Nothing about that 40 ETH sale means ETH became 69% cheaper anywhere in the world. On Coinbase it is still $2,000. This one contract, with 50 ETH in it, now says $617 — and it is not lying, malfunctioning, or hacked. It has never had any other job.

The pool is completely passive. No clock, no loop, no background process. Between calls it is a vending machine with the power off. When the price moves back toward reality, that is not the pool correcting itself. That is arbitrageurs. Their greed does the correcting. The contract just answers the phone.

Remember **$617/ETH** and "a boulder in a puddle". Notebook 7 is where someone actually shoves this puddle — first with a suitcase of cash, then with a suitcase they rented for one elevator ride.


## 3. The lending contract: a loan that asks the puddle

A brilliant judge, sealed in a room, who can reason flawlessly about any evidence presented but cannot leave to gather any. Everything they know arrives through a slot in the door. Corrupt the slot and you do not need to corrupt the judge — the judge will corrupt themselves, impeccably, in full accordance with the law.

Meet the lending protocol. It liquidates whenever collateral value / debt falls below 1.5. That rule is sound, standard, and used by every lending platform on earth.

Its single flaw is one line: it reads `pool.spot_price`. From an AMM. Sitting right there on the same chain, conveniently readable, requiring no external courier, no subscription, no trust in a third party. You can see why someone would do it. It looks like the *more* decentralised choice.

**KEY INSIGHT.** A contract cannot squint at an exchange. So **whatever it treats as the price *is* its oracle** — whether or not anyone called it one, whether or not it appears anywhere in the documentation. If your lending contract reads a number off an AMM, that AMM is your oracle. You made an oracle choice. You may not have noticed you were making one, which is precisely the problem.

The next cell defines `Loan` and `LendingProtocol` — again the same classes notebook 7 will import. Then we value the loan we will keep bullying: **10 ETH collateral, $12,000 debt**. At a sensible ~$2,000/ETH the ratio is 1.67.

> Pause and predict: at $2,000/ETH, is this position above or below a 1.5 threshold?


In [9]:
@dataclass(frozen=True)
class Loan:
    """A borrower's position: collateral posted against debt owed.

    Attributes:
        collateral_eth: ETH locked as collateral. Must be positive.
        debt_usd: Outstanding debt in USD. Must be positive.
    """

    collateral_eth: float
    debt_usd: float

    def __post_init__(self) -> None:
        """Reject non-positive collateral or debt.

        Raises:
            ValueError: If either field is not positive.
        """
        if self.collateral_eth <= 0 or self.debt_usd <= 0:
            raise ValueError("Loan collateral and debt must be positive.")


class LoanNotFoundError(Exception):
    """Raised when a borrower has no open loan."""


class PositionNotLiquidatableError(Exception):
    """Raised when liquidation is attempted on a still-healthy position."""


class LendingProtocol(SmartContract):
    """A toy lending protocol that liquidates undercollateralised loans.

    Its one deliberate flaw: by default it prices collateral from
    ``pool.spot_price`` -- a single, on-chain, manipulable number -- unless
    a ``price_source`` (e.g. a ``MedianOracle``) is supplied instead.
    """

    def __init__(
        self,
        address: str,
        pool: AMMPool,
        liquidation_ratio: float = 1.5,
        price_source=None,
    ) -> None:
        """Configure the protocol's price source and liquidation threshold.

        Args:
            address: On-chain address of this contract.
            pool: The AMM this protocol reads a price from by default.
            liquidation_ratio: Minimum healthy collateral ratio. Must be
                positive.
            price_source: Optional object exposing a ``.price`` property.
                When given, it is trusted instead of ``pool.spot_price``.

        Raises:
            ValueError: If ``liquidation_ratio`` is not positive.
        """
        super().__init__(address)
        if liquidation_ratio <= 0:
            raise ValueError("Liquidation ratio must be positive.")
        self.pool = pool
        self.liquidation_ratio = liquidation_ratio
        self.price_source = price_source
        self.loans: dict[str, Loan] = {}

    def open_loan(
        self, borrower: str, collateral_eth: float, debt_usd: float
    ) -> Loan:
        """Open a loan for ``borrower`` and return it.

        Args:
            borrower: Non-empty borrower identifier.
            collateral_eth: ETH locked as collateral. Must be positive.
            debt_usd: Outstanding debt in USD. Must be positive.

        Returns:
            The registered ``Loan``.
        """
        return self.add_loan(borrower, Loan(collateral_eth, debt_usd))

    def add_loan(self, borrower: str, loan: Loan) -> Loan:
        """Register an open loan for a borrower.

        Args:
            borrower: Non-empty borrower identifier.
            loan: The loan to register.

        Returns:
            The registered ``Loan``.

        Raises:
            ValueError: If ``borrower`` is empty.
        """
        if not borrower:
            raise ValueError("Borrower name must be non-empty.")
        self.loans[borrower] = loan
        return loan

    def collateral_ratio(self, loan: Loan) -> float:
        """Return collateral value divided by debt.

        Reads ``price_source.price`` when one was supplied, otherwise
        ``pool.spot_price``. That read is a contract-to-contract call
        when the source is another contract.

        Args:
            loan: Position to value.

        Returns:
            Collateral value divided by debt at the configured price.
        """
        price = (
            self.price_source.price
            if self.price_source is not None
            else self.pool.spot_price
        )
        return loan.collateral_eth * price / loan.debt_usd

    def liquidate(self, borrower: str) -> Loan:
        """Seize and remove a borrower's loan if it is unhealthy.

        Args:
            borrower: The borrower to liquidate.

        Returns:
            The removed ``Loan``.

        Raises:
            LoanNotFoundError: If ``borrower`` has no open loan.
            PositionNotLiquidatableError: If the loan's collateral ratio
                is still at or above ``self.liquidation_ratio``.
        """
        if borrower not in self.loans:
            raise LoanNotFoundError(f"No loan for {borrower}.")
        loan = self.loans[borrower]
        if self.collateral_ratio(loan) >= self.liquidation_ratio:
            raise PositionNotLiquidatableError("Position is still healthy.")
        return self.loans.pop(borrower)


In [10]:
baseline_pool = AMMPool("amm-baseline", 50.0, 100_000.0)
baseline_protocol = LendingProtocol("lending-baseline", baseline_pool)
victim_loan = baseline_protocol.call(
    "open_loan", borrower="victim", collateral_eth=10.0, debt_usd=12_000.0
)
baseline_ratio = baseline_protocol.collateral_ratio(victim_loan)

print(f"At ${baseline_pool.spot_price:,.2f}/ETH, victim collateral ratio: {baseline_ratio:.2f}")
print("Liquidation threshold: 1.50; verdict: HEALTHY")


At $2,000.00/ETH, victim collateral ratio: 1.67
Liquidation threshold: 1.50; verdict: HEALTHY


**Read the result.** $20,000 of collateral / $12,000 debt is 1.67, so Bob is comfortably healthy. He is not doing anything risky. He is about to be robbed anyway — not in this notebook, in the next one — and he will not have touched a single input.

A later liquidation would not be Bob changing his loan. It would be the protocol trusting a technically true fact ("this pool's ratio is 617") that is a catastrophic answer to the question actually being asked ("what is ETH worth?").


## 4. Put it on the chain we already built

Same `Validator`, `Blockchain`, `Network`, `Transaction`, and `Block` as notebooks 2 and 5. A common mental model is that the contract somehow lives inside the blocks, replicated over and over. It does not. The contract sits at an address. Blocks contain **transactions that call** the contract. That is all.

Exactly the same shape as Merkle roots in notebook 3: the block does not contain 8,000 diplomas, it contains one root pointing at them. Blockchains are relentlessly consistent about this — they store pointers and commitments, and keep the bulky reality somewhere else.

The next cell wires the objects together, then uses `submit_call` four times: deploy `amm.eth`, deploy `lending.eth`, Bob opens the loan, Alice swaps 1 ETH. Inclusion is the notary stamp. `call` is the form being filled in. A node that missed the gossip still cannot include it. Notebook 5 still applies.

> Pause and predict: after Alice's 1 ETH swap, is Bob still above 1.5? And did `lending.eth` have to wait for another transaction before it could read `amm.eth`?


In [13]:
# Same validators / Network / Blockchain as notebooks 2 and 5. One laptop, now with friends.
nodes = ["Node A", "Alice-Node", "Bob-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("Alice-Node", 80),
    Validator("Bob-Node", 70),
    Validator("Farid-Node", 50),
]
network = Network(nodes, random.Random(7))
chain = Blockchain(validators)

# The contract is NOT copied into every block. It sits at an address. Blocks only stamp calls.
amm = AMMPool("amm.eth", 50.0, 100_000.0)
lending = LendingProtocol("lending.eth", amm)

# 'Deploy' here is a receipt, not bytecode. The objects already exist in memory. The notary files the form.
network.broadcast(Transaction("tx-deploy-amm", "Deploy AMMPool at amm.eth"), origin="Node A")
network.broadcast(Transaction("tx-deploy-lending", "Deploy LendingProtocol at lending.eth"), origin="Node A")
_, deploy_amm_block = network.include("Node A", "tx-deploy-amm", chain)
_, deploy_lending_block = network.include("Node A", "tx-deploy-lending", chain)

# Bob's loan, then Alice's pebble. Two shouts into the group chat, two stamps.
bob_loan, bob_tx, bob_block = submit_call(
    network,
    chain,
    "Bob-Node",
    "tx-open-loan",
    lending,
    "open_loan",
    borrower="Bob",
    collateral_eth=10.0,
    debt_usd=12_000.0,
)
alice_usd, alice_tx, alice_block = submit_call(
    network,
    chain,
    "Alice-Node",
    "tx-swap-1eth",
    amm,
    "swap_eth_for_usd",
    eth_in=1.0,
)
bob_ratio_after_pebble = lending.collateral_ratio(lending.loans["Bob"])
valid, message = chain.is_valid()

print(f"Deployed {amm.address} in Block #{deploy_amm_block.index}")
print(f"Deployed {lending.address} in Block #{deploy_lending_block.index}")
print(f"{bob_block.proposer} included {bob_tx.tx_id} in Block #{bob_block.index}")
print(f"{alice_block.proposer} included {alice_tx.tx_id} in Block #{alice_block.index}")
print(
    f"After Alice's 1 ETH swap: pool spot ${amm.spot_price:,.2f}/ETH; "
    f"Bob's ratio {bob_ratio_after_pebble:.2f}"
)
print(f"Chain valid? {valid} -- {message}")
print(f"Canonical length: {len(chain.chain)} (genesis + {len(chain.chain) - 1} inclusions)")


Deployed amm.eth in Block #1
Deployed lending.eth in Block #2
Bob-Node included tx-open-loan in Block #3
Alice-Node included tx-swap-1eth in Block #4
After Alice's 1 ETH swap: pool spot $1,922.34/ETH; Bob's ratio 1.60
Chain valid? True -- Chain is valid.
Canonical length: 5 (genesis + 4 inclusions)


### Open the `Block` objects

`include` already returned `Block` instances — the same class as notebook 2. We just have not looked inside them yet. Walk the chain. Every payload is a call. None of the payloads *is* the contract. If you want the contract, you look up the address. If you want the history of calls, you walk `chain.chain`.


In [15]:
# Every item in chain.chain is the same Block class as notebook 2. No new species of block.
assert all(isinstance(block, Block) for block in chain.chain)

print("=== Each Block records a call. None of them stores the contract. ===")
for block in chain.chain:
    print(block)

# Still at their addresses. The blocks stored receipts, not the vending machines.
print(f"{amm.address} still lives at one address.")
print(f"  current reserves: {amm.eth_reserve:.2f} ETH / ${amm.usd_reserve:,.2f}")
print(f"{lending.address} still lives at one address.")
print(f"  open loans: {list(lending.loans)}")
print(
    f"Block #{alice_block.index} data is only the receipt: {alice_block.data!r}"
)
print("A later block can call these addresses. They were not copied into every block.")

# lending.eth asking amm.eth is an inner phone call, not a new Transaction. No extra stamp.
print()
print("=== Contract-to-contract: no extra Block ===")
print(f"{lending.address} asks {amm.address} for spot_price.")
print(f"  {amm.address}.spot_price = ${amm.spot_price:,.2f}/ETH")
print(f"  {lending.address} computes Bob's ratio as {bob_ratio_after_pebble:.2f}")
print(
    f"That inner read did not append a Block. Chain length is still "
    f"{len(chain.chain)}."
)


=== Each Block records a call. None of them stores the contract. ===
Block #0 proposed by network
  data:          Genesis Block
  previous_hash: 0000000000000000...
  hash:          02ee86b59214716d...

Block #1 proposed by Node A
  data:          Deploy AMMPool at amm.eth
  previous_hash: 02ee86b59214716d...
  hash:          25c0ad24cb350b7e...

Block #2 proposed by Node A
  data:          Deploy LendingProtocol at lending.eth
  previous_hash: 25c0ad24cb350b7e...
  hash:          13a068784d9336a2...

Block #3 proposed by Bob-Node
  data:          lending.eth.open_loan(borrower='Bob', collateral_eth=10.0, debt_usd=12000.0)
  previous_hash: 13a068784d9336a2...
  hash:          a82bd48090b233e7...

Block #4 proposed by Alice-Node
  data:          amm.eth.swap_eth_for_usd(eth_in=1.0)
  previous_hash: a82bd48090b233e7...
  hash:          167f46eb5a83c199...

amm.eth still lives at one address.
  current reserves: 51.00 ETH / $98,039.22
lending.eth still lives at one address.
  open loans:

**Read the result.** Bob's loan and Alice's swap are ordinary notebook-5 transactions. Each one produced a `Block`. Alice's pebble moves the puddle a little (~$1,922/ETH) and Bob stays healthy at about 1.60.

The contract is **not** in every block. Block #1 recorded the deploy. Block #4 recorded Alice's swap. `amm.eth` still lives at one address; its reserves changed because the call ran, not because the block swallowed the pool.

`lending.eth` can talk to `amm.eth`. It already did: computing Bob's ratio is a contract-to-contract read of `spot_price`. That inner call did not append a Block. External transactions get stamps. Internal calls happen while a stamp is being filled in.

**CLOWN PROBLEM.** Every one of those benefits — public addresses, permissionless calls, money Legos — is also the attack surface. If your contract reads a price from another contract, whoever can shove that other contract controls your decisions. The Lego blocks snap together beautifully, and they also snap together with the ones a thief brought.


## 5. Oracles, or the chain cannot see the weather

A smart contract is deterministic. It has to be — every node independently re-executes it and must reach the same answer. Determinism forbids HTTP requests, files, clocks, random numbers, and looking at literally anything outside its own storage and its arguments.

**KEY INSIGHT.** A smart contract is a brain in a jar. It cannot find out the price of ETH, the weather in Singapore, who won the election, or whether a shipping container arrived. Not "it is difficult" — it is structurally impossible, and it is impossible for the **same reason** that makes the chain trustworthy. Determinism giveth and determinism taketh away.

So somebody has to *tell* it. An oracle is not a special network feature, not a field in the block header. It is just another smart contract, at its own address, with its own storage. Same shape as `AMMPool`. Same shape as `LendingProtocol`. Just a fourth box at a fourth address.

The next cell defines `PriceReport`, `MedianOracle`, and `PriceFeed` — notebook 7 will import them unchanged. The cell after that deploys `feed.eth` and stamps three reports: $2,005, then a $1,200 lie, then $1,995. Each in its own `Block`. Different blocks, different numbers. The lending rule has to choose which one to believe.

A stamped report is *not* authenticated (a `source` string is a label, not a signature), *not* fresh (this toy has no heartbeat), and *not* trustworthy just because it is on-chain. Readable is not the same as "the market."

> Pause and predict: if `lending.eth` trusts only the latest block's report, what happens when that block is the $1,200 lie? What if it waits and takes the median of all three reports?


In [18]:
@dataclass(frozen=True)
class PriceReport:
    """One reported ETH/USD price from a named source.

    Attributes:
        source: Who reported this price. Nothing here authenticates them.
        eth_price_usd: The reported price.
    """

    source: str
    eth_price_usd: float


def collateral_ratio(
    collateral_eth: float, debt_usd: float, eth_price_usd: float
) -> float:
    """Return collateral value divided by debt, at a given ETH price.

    Args:
        collateral_eth: ETH locked as collateral. Must be positive.
        debt_usd: Outstanding debt in USD. Must be positive.
        eth_price_usd: ETH/USD price the caller is treating as fact.

    Returns:
        Collateral value divided by debt.

    Raises:
        ValueError: If any input is not positive.
    """
    if collateral_eth <= 0 or debt_usd <= 0 or eth_price_usd <= 0:
        raise ValueError("Collateral, debt, and price must be positive.")
    return collateral_eth * eth_price_usd / debt_usd


def should_liquidate(ratio: float, threshold: float = 1.5) -> bool:
    """Return whether a collateral ratio is below the liquidation threshold.

    Args:
        ratio: Collateral value divided by debt.
        threshold: Minimum healthy ratio. Defaults to 1.5.

    Returns:
        True if ``ratio`` is strictly below ``threshold``.
    """
    return ratio < threshold


class MedianOracle:
    """A price source that reports the median of several independent reports."""

    def __init__(self, reports: list[PriceReport]) -> None:
        """Store the reports this oracle will median.

        Args:
            reports: Non-empty list of named price reports.

        Raises:
            ValueError: If ``reports`` is empty.
        """
        if not reports:
            raise ValueError("At least one price report is required.")
        self.reports = reports

    @property
    def price(self) -> float:
        """Return the median reported price.

        Returns:
            Median of ``eth_price_usd`` across ``self.reports``.
        """
        return statistics.median(report.eth_price_usd for report in self.reports)


class PriceFeed(SmartContract):
    """A contract that stores price reports as they arrive.

    Each ``report`` call is meant to be stamped into its own Block.
    ``latest`` is whatever the last stamp said. A ``MedianOracle`` over
    ``reports`` is a different choice.
    """

    def __init__(self, address: str) -> None:
        """Create an empty feed at ``address``.

        Args:
            address: On-chain address of this feed.
        """
        super().__init__(address)
        self.reports: list[PriceReport] = []

    def report(self, source: str, eth_price_usd: float) -> PriceReport:
        """Append one report and return it.

        Args:
            source: Label for who sent this number. Not a signature.
            eth_price_usd: Reported ETH/USD price. Must be positive.

        Returns:
            The stored ``PriceReport``.

        Raises:
            ValueError: If ``eth_price_usd`` is not positive.
        """
        if eth_price_usd <= 0:
            raise ValueError("Reported price must be positive.")
        posted = PriceReport(source, eth_price_usd)
        self.reports.append(posted)
        return posted

    @property
    def latest(self) -> PriceReport:
        """Return the most recently stamped report.

        Returns:
            The last item in ``self.reports``.

        Raises:
            IndexError: If no report has been posted yet.
        """
        return self.reports[-1]

    @property
    def price(self) -> float:
        """Return the median of every report stored so far.

        Returns:
            Median ETH/USD price across ``self.reports``.
        """
        return statistics.median(report.eth_price_usd for report in self.reports)


In [19]:
# Fourth box, fourth address. Deploy is still just a receipt. No magic oracle opcode.
feed = PriceFeed("feed.eth")
network.broadcast(
    Transaction("tx-deploy-feed", "Deploy PriceFeed at feed.eth"), origin="Node A"
)
_, deploy_feed_block = network.include("Node A", "tx-deploy-feed", chain)
assert isinstance(deploy_feed_block, Block)

collateral_eth = 10
debt_usd = 12_000
# Honest, liar, honest. Each shout becomes its own Block. The lie gets a stamp too.
incoming_reports = [
    ("tx-report-a", "independent-feed-a", 2_005.0),
    ("tx-report-lie", "malicious-feed", 1_200.0),
    ("tx-report-b", "independent-feed-b", 1_995.0),
]

print(f"Same loan: {collateral_eth} ETH / ${debt_usd:,} debt. Threshold 1.50.")
print(f"Deployed {feed.address} in Block #{deploy_feed_block.index}")
print()

for tx_id, source, price in incoming_reports:
    _, tx, block = submit_call(
        network,
        chain,
        "Node A",
        tx_id,
        feed,
        "report",
        source=source,
        eth_price_usd=price,
    )
    assert isinstance(block, Block)
    # Trusting only latest makes the $1,200 lie look like THE price. The notary does not fact-check.
    latest_ratio = collateral_ratio(
        collateral_eth, debt_usd, feed.latest.eth_price_usd
    )
    latest_verdict = (
        "LIQUIDATE (wrong)" if should_liquidate(latest_ratio) else "HEALTHY"
    )
    print(block)
    print(
        f"  latest on {feed.address}: ${feed.latest.eth_price_usd:,.0f}/ETH "
        f"from {feed.latest.source}"
    )
    print(
        f"  if lending trusts only Block #{block.index}: "
        f"ratio {latest_ratio:.2f} -> {latest_verdict}"
    )
    print()

# Median: sort three numbers, pick the middle one. No ML. No consensus. Dumber than the name implies, and that is the point.
median_oracle = MedianOracle(feed.reports)
median_ratio = collateral_ratio(collateral_eth, debt_usd, median_oracle.price)
print(
    f"Median of every report stored at {feed.address}: "
    f"${median_oracle.price:,.0f}/ETH"
)
print(
    f"If lending reads the median instead of the latest Block: "
    f"ratio {median_ratio:.2f} -> HEALTHY"
)
print(
    f"Chain length: {len(chain.chain)}. "
    "The three reports are still sitting on their own Blocks."
)


Same loan: 10 ETH / $12,000 debt. Threshold 1.50.
Deployed feed.eth in Block #5

Block #6 proposed by Node A
  data:          feed.eth.report(source='independent-feed-a', eth_price_usd=2005.0)
  previous_hash: 90d4b6f262ea07ff...
  hash:          188772a36710b029...

  latest on feed.eth: $2,005/ETH from independent-feed-a
  if lending trusts only Block #6: ratio 1.67 -> HEALTHY

Block #7 proposed by Node A
  data:          feed.eth.report(source='malicious-feed', eth_price_usd=1200.0)
  previous_hash: 188772a36710b029...
  hash:          d7089ce65d1d1518...

  latest on feed.eth: $1,200/ETH from malicious-feed
  if lending trusts only Block #7: ratio 1.00 -> LIQUIDATE (wrong)

Block #8 proposed by Node A
  data:          feed.eth.report(source='independent-feed-b', eth_price_usd=1995.0)
  previous_hash: d7089ce65d1d1518...
  hash:          90c9740cc5c0ef63...

  latest on feed.eth: $1,995/ETH from independent-feed-b
  if lending trusts only Block #8: ratio 1.66 -> HEALTHY

Median of e

**Read the result.** Three reports, three `Block`s, three different numbers. When the $1,200 lie is the latest stamp, a lending rule that trusts only the tip liquidates a healthy loan. The next block arrives at $1,995 and the same rule flips back to healthy. Different blocks, different oracles — if "oracle" means "whatever this contract just read."

The median of all three stays near $1,995 and the ratio 1.66 stays healthy. Why the median rather than the average? Because an average is hostage to any single outlier. The median just picks the middle value and ignores how extreme the liar was.

**KEY INSIGHT.** Aggregation did not make the AMM honest — the AMM was always honest. The defence is changing which question you ask, not improving the answer. And "we use multiple oracles" is not a security property. It is the *shape* of a security property. Five oracles that all scrape the same exchange are one oracle wearing five hats.

**BULLSHIT DETECTOR.** The lending protocol's authors probably felt clever. "We do not depend on any external price provider, everything is on-chain and trustless." They eliminated a trusted third party and replaced it with a number that anyone with a suitcase of cash can set to whatever they want, for one transaction, for a fee. That is not removing trust. That is relocating it.

On-chain readable is not the same as trustworthy. A puddle is not the ocean. A valid `Block` is not a journalist.

Notebook 7 is where Alice shoves the puddle. Flash loans, in that notebook, hand anyone the boulder for one transaction.


## Takeaways

- **Mechanism:** a smart contract lives at an address. **Not a guarantee:** a copy of it sits inside every `Block`.
- **Mechanism:** the **code** is frozen at deployment. **Not a guarantee:** the **state** is frozen. The numbers move constantly.
- **Mechanism:** one contract can call another while a transaction is being filled in. **Not a guarantee:** that inner read appends its own block, or that the callee was a good price source.
- **Mechanism:** `broadcast`, then `include`, then `call` — same waiting room as notebook 5. **Not a guarantee:** a missed gossip still lets you include it. **Inclusion is not success.**
- **Mechanism:** an AMM spot price honestly reflects its own reserves. **Not a guarantee:** that local ratio is "the market." A puddle is not the ocean.
- **Mechanism:** whatever a contract treats as fact **is** its oracle. **Not a guarantee:** you noticed you were making an oracle choice, or that a median of stamps is true.

The two sentences to tattoo: agreement does not create information. And whatever a contract treats as fact is its oracle.

Next: [7. flash_loans.ipynb](7.%20flash_loans.ipynb) — same contracts, now shoved. First with owned ETH. Then with a flash loan. Every step legal. Bob loses his house.


## Sources

Contracts at addresses, AMMs, and oracles as *inputs a contract treats as fact* all have homes outside this notebook:

- Szabo, N. (1997). [The Idea of Smart Contracts](https://nakamotoinstitute.org/library/the-idea-of-smart-contracts/). Code that fills in an agreement when named conditions hold — a vending machine, not a lawyer.
- Buterin, V. (2014). [Ethereum White Paper](https://ethereum.org/en/whitepaper/). Accounts, contract code, and calls from one contract to another on a public chain.
- Adams, H., Zinsmeister, N., & Robinson, D. (2020). [Uniswap v2 Core](https://uniswap.org/whitepaper.pdf). Constant-product pools (`x * y = k`) and why the displayed price is just the reserve ratio.
- Ellis, S., Juels, A., & Nazarov, S. (2017). [ChainLink: A Decentralized Oracle Network](https://research.chain.link/whitepaper-v1.pdf). Why a contract cannot see an exchange screen, and why one report is a different choice from a median of reports.
